# ROUGE and BERTScore for baseline vs BRIO checkpoints, on both Liputan6 partitions

BERTScore compares the meaning of a summary with the reference summary using BERT embeddings, so it complements ROUGE (word overlap). `bertscore_eval.py` computes both, per article, so every comparison with the baseline gets a confidence interval and a sign test.

- **Part A** runs on the 100 articles used for the Gemini judge. Quick.
- **Part B** generates summaries for the whole Canonical test set (10,972 articles).
- **Part C** generates summaries for the whole XTreme test set (3,862 articles).
- **Scoring** evaluates both partitions in one run and writes one report.

Several BRIO checkpoints can be evaluated in one run: list them in `BRIO_CKPTS` in the Paths cell. Each one is compared with the baseline.

In [1]:
# Install condacolab. The runtime restarts after this cell, that is normal.
!pip install -q condacolab
import condacolab
condacolab.install()


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to:

    !pip install -q 'condacolab==0.1.*'

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-26.3.2-3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:10
🔁 Restarting kernel...


### Env

In [1]:
# Clone the repo and mount Google Drive.
!git clone https://github.com/alvrian/BRIO.git

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

%cd BRIO

Cloning into 'BRIO'...
remote: Enumerating objects: 620, done.
remote: Counting objects: 100% (329/329), done.
remote: Compressing objects: 100% (220/220), done.
remote: Total 620 (delta 235), reused 159 (delta 105), pack-reused 291 (from 2)
Receiving objects: 100% (620/620), 16.19 MiB | 27.26 MiB/s, done.
Resolving deltas: 100% (389/389), done.
Mounted at /content/drive
/content/BRIO


In [2]:
# Same env setup as BRIO_1_1.ipynb, plus the bert-score package and the fine-tuned IndoBART baseline.
! conda create --name env --file spec-file.txt
! conda run --no-capture-output -n env pip install -r requirements.txt
! conda run -n env python -c "import nltk;nltk.download('punkt')"
! git clone https://github.com/neulab/compare-mt.git
! cd compare-mt && conda run --no-capture-output -n env pip install -r requirements.txt
! cd compare-mt && conda run -n env python setup.py install
! conda run --no-capture-output -n env pip install bert-score
print("done setting up env")
! cp /content/drive/MyDrive/BRIO/indobart-liputan6-finetuned.zip .
! unzip -q indobart-liputan6-finetuned.zip
! rm indobart-liputan6-finetuned.zip
print("all done")

Streaming output truncated to the last 5000 lines.




libgomp-9.3.0        | :   5% 0.051509378202830754/1 [00:00<00:04,  4.73s/it]



ld_impl_linux-64-2.3 | : 100% 1.0/1 [00:00<00:00,  7.96s/it]                 





libgomp-9.3.0        | : 100% 1.0/1 [00:00<00:00,  4.73s/it]                 




libstdcxx-ng-9.3.0   | :   1% 0.005042836718770439/1 [00:00<00:55, 55.58s/it]



ld_impl_linux-64-2.3 | : 100% 1.0/1 [00:00<00:00,  7.96s/it]







_openmp_mutex-4.5    | :  74% 0.7391833972479134/1 [00:00<00:00,  2.62it/s]







_openmp_mutex-4.5    | : 100% 1.0/1 [00:00<00:00,  2.62it/s]               








libgcc-ng-9.3.0      | :   0% 0.003253736862372134/1 [00:00<01:30, 90.69s/it]


intel-openmp-2021.3. | :   1% 0.011373204801680435/1 [00:00<00:27, 27.88s/it]





libgomp-9.3.0        | : 100% 1.0/1 [00:00<00:00,  4.73s/it]






mkl-2021.3.0         | :   0% 0.00011065242687537004/1 [00:00<51:17, 3077.52s/it]







_openmp_mutex-4.5    | : 100% 1.0/1 [00:00<00:00,  2.62it/s]


i

### Paths

In [ ]:
# List every model_generation.bin saved on Drive, to copy the right paths into the next cell.
import glob
for p in sorted(glob.glob("/content/drive/MyDrive/BRIO/checkpoints/*/*/model_generation.bin")):
    print(p)

/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_0-1/4000/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_0-1_second_seed/master_1_0/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_0-1_third_seed/master_1_0/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-1/4000/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-1/8000/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-100_12/master_1_100/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-100_8/10000/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-100_8/12000/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-100_8/2000/model_generation.bin
/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-100_8/4000/model

In [3]:
import os

# Liputan6 canonical test set on Drive.
TEST_SOURCE = "/content/drive/MyDrive/BRIO/liputan6_candidates/canonical/test.source"
TEST_TARGET = "/content/drive/MyDrive/BRIO/liputan6_candidates/canonical/test.target"

# Fine-tuned IndoBART baseline, unzipped by the env cell.
BASELINE = "./indobart-liputan6-finetuned"

# BRIO checkpoints to evaluate: short name -> path of model_generation.bin.
# EDIT: add or remove lines. Keep the name "brio" for the checkpoint you used with the Gemini judge,
# so its saved summaries are reused.
CKPT_DIR = "/content/drive/MyDrive/BRIO/checkpoints"


BRIO_CKPTS = {
    "brio_1_1_seed1": CKPT_DIR +  "/BRIO_Liputan6_checkpoint_1-1/8000/model_generation.bin",
    "brio_1_1_seed2":   CKPT_DIR + "/BRIO_Liputan6_checkpoint_1-1_second_seed/master_1_1/model_generation.bin",
    "brio_1_1_seed3":   CKPT_DIR + "/BRIO_Liputan6_checkpoint_1-1_third_seed/master_1_1/model_generation.bin",
    "brio_10_1_seed1":  CKPT_DIR +  "/BRIO_Liputan6_checkpoint_10-1-reduceLR/8000/model_generation.bin",
    "brio_10_1_seed2":  CKPT_DIR +  "/BRIO_Liputan6_checkpoint_10-1_second_seed/master_1_10/model_generation.bin",
    "brio_10_1_seed3":  CKPT_DIR +  "/BRIO_Liputan6_checkpoint_10-1_third_seed/master_1_10/model_generation.bin",
    "brio_100_1_seed1": CKPT_DIR +  "/BRIO_Liputan6_checkpoint_100-1-reduceLR/4000/model_generation.bin",
    "brio_100_1_seed2": CKPT_DIR +  "/BRIO_Liputan6_checkpoint_100-1-second_seed/master_1_100/model_generation.bin",
    "brio_100_1_seed3": CKPT_DIR +  "/BRIO_Liputan6_checkpoint_100-1-third_seed/master_1_100/model_generation.bin",
    "brio_1_1_seed1_12cand" : CKPT_DIR + '/BRIO_Liputan6_checkpoint_1-1_12/master_1_1/model_generation.bin',
    "brio_1_1_seed2_12cand" : CKPT_DIR + '/BRIO_Liputan6_checkpoint_1-1_12_second_seed/master_1_1/model_generation.bin',
    "brio_10_1_seed1_12cand" : CKPT_DIR + '/BRIO_Liputan6_checkpoint_1-10_12/master_1_10/model_generation.bin',
    "brio_10_1_seed2_12cand" : CKPT_DIR + '/BRIO_Liputan6_checkpoint_1-10_12_second_seed/master_1_10/model_generation.bin',
    "brio_10_1_seed3_12cand" : CKPT_DIR + '/BRIO_Liputan6_checkpoint_1-10_12_third_seed/master_1_10/model_generation.bin',
    "brio_100_1_seed3_12cand" : CKPT_DIR + '/BRIO_Liputan6_checkpoint_1-100_12/master_1_100/model_generation.bin',
    "brio_1_1_seed1_8cand" : CKPT_DIR +'/BRIO_Liputan6_checkpoint_1-1_8/master_1_1/model_generation.bin',
    "brio_1_1_seed2_8cand" : CKPT_DIR + '/BRIO_Liputan6_checkpoint_1-1_8_second_seed/master_1_1/model_generation.bin',
    "brio_1_1_seed3_8cand" : CKPT_DIR + '/BRIO_Liputan6_checkpoint_1-1_8_third_seed/master_1_1/model_generation.bin',
    "brio_10_1_seed1_8cand" : CKPT_DIR + '/BRIO_Liputan6_checkpoint_1-10_8/master_1_10/model_generation.bin',
    "brio_10_1_seed2_8cand" : CKPT_DIR + '/BRIO_Liputan6_checkpoint_1-10_8_second_seed/8000/model_generation.bin',
    "brio_10_1_seed3_8cand" : CKPT_DIR + '/BRIO_Liputan6_checkpoint_1-10_8_third_seed/master_1_10/model_generation.bin',
    "brio_100_1_seed1_8cand" : CKPT_DIR + '/BRIO_Liputan6_checkpoint_1-100_8/master_1_100/model_generation.bin',
    "brio_1_1_seed1_32cand" : CKPT_DIR +  '/BRIO/Liputan6_32c_6000_1-1_seed970903/maseter_1_1/model_generation.bin',
    "brio_1_1_seed3_32cand" : CKPT_DIR +  '/BRIO/Liputan6_32c_6000_1-1_seed70905/maseter_1_1/model_generation.bin',
    "brio_1_1_seed3_32cand" : CKPT_DIR +  '/BRIO/Liputan6_32c_6000_1-1_seed81215/maseter_1_1/model_generation.bin',
    "brio_10_1_seed1_32cand" : CKPT_DIR +  '/BRIO/Liputan6_32c_6000_1-10_seed970903/maseter_1_10/model_generation.bin',
    "brio_10_1_seed3_32cand" : CKPT_DIR +  '/BRIO/Liputan6_32c_6000_1-10_seed70905/maseter_1_10/model_generation.bin',
    "brio_10_1_seed3_32cand" : CKPT_DIR +  '/BRIO/Liputan6_32c_6000_1-10_seed81215/maseter_1_10/model_generation.bin',
    "brio_100_1_seed1_32cand" : CKPT_DIR +  '/BRIO/Liputan6_32c_6000_1-100_seed970903/maseter_1_100/model_generation.bin',
    "brio_100_1_seed3_32cand" : CKPT_DIR +  '/BRIO/Liputan6_32c_6000_1-100_seed70905/maseter_1_100/model_generation.bin',
    "brio_100_1_seed3_32cand" : CKPT_DIR +  '/BRIO/Liputan6_32c_6000_1-100_seed81215/maseter_1_100/model_generation.bin',
}

# Folder with the 100-article sample (Part A) and a folder for the whole test set (Part B).
SAMPLE_DIR = "/content/drive/MyDrive/BRIO/eval_outputs"
FULL_DIR = "/content/drive/MyDrive/BRIO/eval_outputs_full"        # Canonical test set
XTREME_DIR = "/content/drive/MyDrive/BRIO/eval_outputs_xtreme"    # XTreme test set
REPORT_DIR = "/content/drive/MyDrive/BRIO/eval_report"             # combined report for both partitions

for p in [TEST_SOURCE, TEST_TARGET, BASELINE] + list(BRIO_CKPTS.values()):
    print("found  " if os.path.exists(p) else "MISSING", p)

# Build the system list and the baseline-vs-checkpoint pairs once, used by every scoring cell.
SYSTEMS = " ".join(["baseline"] + list(BRIO_CKPTS))
PAIRS = " ".join("--pair baseline " + name for name in BRIO_CKPTS)
print(SYSTEMS)

found   /content/drive/MyDrive/BRIO/liputan6_candidates/canonical/test.source
found   /content/drive/MyDrive/BRIO/liputan6_candidates/canonical/test.target
found   ./indobart-liputan6-finetuned
found   /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-1/8000/model_generation.bin
found   /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-1_second_seed/master_1_1/model_generation.bin
found   /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-1_third_seed/master_1_1/model_generation.bin
found   /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_10-1-reduceLR/8000/model_generation.bin
found   /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_10-1_second_seed/master_1_10/model_generation.bin
found   /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_10-1_third_seed/master_1_10/model_generation.bin
found   /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_100-1-reduceLR/4000/model_gener

In [4]:
# Generate and save the summaries of the baseline and every checkpoint into one folder.
# sample_n = 0 uses the whole test set. src / tgt default to the Canonical test files.
# A model whose summaries are already complete in that folder is skipped.
def generate_all(out_dir, sample_n, batch_size=8, src=None, tgt=None):
    src = src or TEST_SOURCE
    tgt = tgt or TEST_TARGET
    models = {"baseline": BASELINE}
    models.update(BRIO_CKPTS)
    for name, path in models.items():
        print("=====", name)
        !conda run --no-capture-output -n env python generate_outputs.py --name {name} --model_path "{path}" --src_file "{src}" --tgt_file "{tgt}" --out_dir "{out_dir}" --sample_n {sample_n} --batch_size {batch_size}


### Part A: the 100 sampled articles

In [ ]:
# Summaries for the 100 sampled articles (only the checkpoints not saved yet are generated).
generate_all(SAMPLE_DIR, 1)

In [ ]:
# BERTScore for every system on the 100 articles. Results are saved in SAMPLE_DIR/bertscore/.
!conda run --no-capture-output -n env python bertscore_eval.py --data_dir "{SAMPLE_DIR}" --report_dir "{SAMPLE_DIR}" --systems {SYSTEMS} {PAIRS}

### Part B: whole test set

Generation is the slow part and the time grows with every checkpoint in the list. If Colab disconnects, run the same cell again: finished models are skipped and the interrupted one continues from its last saved summary.

In [5]:
# Summaries for all 10,972 test articles, for the baseline and every checkpoint.
# generate_all(FULL_DIR, 0)
generate_all(FULL_DIR, 0, batch_size=16)

===== baseline
Reusing 10972 article ids from /content/drive/MyDrive/BRIO/eval_outputs_full/sample_ids.txt
/content/drive/MyDrive/BRIO/eval_outputs_full/baseline.out already has all 10972 summaries, nothing to generate
Saved 10972 summaries to /content/drive/MyDrive/BRIO/eval_outputs_full/baseline.out
ROUGE-1 0.3937 | ROUGE-2 0.2165 | ROUGE-L 0.3649  (n=10972)
===== brio_1_1_seed1
Reusing 10972 article ids from /content/drive/MyDrive/BRIO/eval_outputs_full/sample_ids.txt
/content/drive/MyDrive/BRIO/eval_outputs_full/brio_1_1_seed1.out already has all 10972 summaries, nothing to generate
Saved 10972 summaries to /content/drive/MyDrive/BRIO/eval_outputs_full/brio_1_1_seed1.out
ROUGE-1 0.4119 | ROUGE-2 0.2274 | ROUGE-L 0.3814  (n=10972)
===== brio_1_1_seed2
Reusing 10972 article ids from /content/drive/MyDrive/BRIO/eval_outputs_full/sample_ids.txt
/content/drive/MyDrive/BRIO/eval_outputs_full/brio_1_1_seed2.out already has all 10972 summaries, nothing to generate
Saved 10972 summaries to 

### Part C: XTreme test set

The XTreme partition uses the same trained models; only the test articles differ. Its test files are taken from Drive if they are there, otherwise from `liputan6_converted.zip`.

In [6]:
# Find the XTreme test files: on Drive first, otherwise unzip the converted dataset and use its xtreme folder.
import glob
XTREME_SOURCE = "/content/drive/MyDrive/BRIO/liputan6_candidates/xtreme/test.source"
XTREME_TARGET = "/content/drive/MyDrive/BRIO/liputan6_candidates/xtreme/test.target"
if not (os.path.exists(XTREME_SOURCE) and os.path.exists(XTREME_TARGET)):
    if not glob.glob("./**/xtreme/test.source", recursive=True):
        !cp /content/drive/MyDrive/BRIO/Dataset/liputan6_converted.zip .
        !unzip -q -o liputan6_converted.zip
    found = sorted(glob.glob("./**/xtreme/test.source", recursive=True))
    if not found:
        raise FileNotFoundError("No xtreme/test.source found: put the XTreme test files on Drive and set XTREME_SOURCE / XTREME_TARGET")
    XTREME_SOURCE = found[0]
    XTREME_TARGET = found[0].replace(".source", ".target")
for p in [XTREME_SOURCE, XTREME_TARGET]:
    n = sum(1 for _ in open(p, encoding="utf-8")) if os.path.exists(p) else 0
    print("found  " if n else "MISSING", p, "(%d lines)" % n)   # expect 3862 lines each

found   /content/drive/MyDrive/BRIO/liputan6_candidates/xtreme/test.source (3862 lines)
found   /content/drive/MyDrive/BRIO/liputan6_candidates/xtreme/test.target (3862 lines)


In [ ]:
# Summaries for all 3,862 XTreme test articles, for the baseline and every checkpoint.
generate_all(XTREME_DIR, 0, batch_size=16, src=XTREME_SOURCE, tgt=XTREME_TARGET)

===== baseline
Reusing 3862 article ids from /content/drive/MyDrive/BRIO/eval_outputs_xtreme/sample_ids.txt
/content/drive/MyDrive/BRIO/eval_outputs_xtreme/baseline.out already has all 3862 summaries, nothing to generate
Saved 3862 summaries to /content/drive/MyDrive/BRIO/eval_outputs_xtreme/baseline.out
ROUGE-1 0.3330 | ROUGE-2 0.1402 | ROUGE-L 0.3005  (n=3862)
===== brio_1_1_seed1
Reusing 3862 article ids from /content/drive/MyDrive/BRIO/eval_outputs_xtreme/sample_ids.txt
/content/drive/MyDrive/BRIO/eval_outputs_xtreme/brio_1_1_seed1.out already has all 3862 summaries, nothing to generate
Saved 3862 summaries to /content/drive/MyDrive/BRIO/eval_outputs_xtreme/brio_1_1_seed1.out
ROUGE-1 0.3549 | ROUGE-2 0.1553 | ROUGE-L 0.3212  (n=3862)
===== brio_1_1_seed2
Reusing 3862 article ids from /content/drive/MyDrive/BRIO/eval_outputs_xtreme/sample_ids.txt
/content/drive/MyDrive/BRIO/eval_outputs_xtreme/brio_1_1_seed2.out already has all 3862 summaries, nothing to generate
Saved 3862 summarie

### Scoring: both partitions

One run scores ROUGE and BERTScore for every system on both partitions. Scores already saved are reused, so a rerun after adding a checkpoint only scores the new one. A system whose summaries are not complete stops the run with a message; add `--skip_missing` to leave it out instead.

Results are written to `REPORT_DIR`: `report_<model>.txt` (tables per partition, mean ± SD over seeds per setting, comparisons with the baseline), `results_<model>.csv` and `results_<model>_by_setting.csv`. `<model>` is `mbert` here, so the IndoBERT run below writes its own files next to these.

In [ ]:
# ROUGE and BERTScore (multilingual BERT) on both partitions.
!conda run --no-capture-output -n env python bertscore_eval.py --data_dir canonical="{FULL_DIR}" --data_dir xtreme="{XTREME_DIR}" --report_dir "{REPORT_DIR}" --systems {SYSTEMS} {PAIRS}

In [ ]:
# Show the combined report, and the per-setting table (mean and SD over seeds) as a dataframe.
import pandas as pd
print(open(os.path.join(REPORT_DIR, "report_mbert.txt"), encoding="utf-8").read())
by_setting = pd.read_csv(os.path.join(REPORT_DIR, "results_mbert_by_setting.csv"))
by_setting

### Optional: IndoBERT instead of multilingual BERT

The same scoring with IndoBERT embeddings, to check that the result does not depend on the embedding model. It writes `*_indobert-base-p1*` files and does not overwrite the multilingual BERT results. ROUGE is reused from the run above.

In [ ]:
# ROUGE and BERTScore (IndoBERT, layer 9) on both partitions.
!conda run --no-capture-output -n env python bertscore_eval.py --data_dir canonical="{FULL_DIR}" --data_dir xtreme="{XTREME_DIR}" --report_dir "{REPORT_DIR}" --systems {SYSTEMS} {PAIRS} --model_type indobenchmark/indobert-base-p1 --num_layers 9

In [ ]:
import os
CKPT_DIR = "/content/drive/MyDrive/BRIO/checkpoints"

# BRIO_CKPTS = {
#     "brio_1_1_seed1": CKPT_DIR +  "/BRIO_Liputan6_checkpoint_1-1/8000/model_generation.bin",
#     "brio_1_1_seed2":   CKPT_DIR + "/BRIO_Liputan6_checkpoint_1-1_second_seed/master_1_1/model_generation.bin",
#     "brio_1_1_seed3":   CKPT_DIR + "/BRIO_Liputan6_checkpoint_1-1_third_seed/master_1_1/model_generation.bin",
#     "brio_10_1_seed1":  CKPT_DIR +  "/BRIO_Liputan6_checkpoint_10-1-reduceLR/8000/model_generation.bin",
#     "brio_10_1_seed2":  CKPT_DIR +  "/BRIO_Liputan6_checkpoint_10-1_second_seed/master_1_10/model_generation.bin",
#     "brio_10_1_seed3":  CKPT_DIR +  "/BRIO_Liputan6_checkpoint_10-1_third_seed/master_1_10/model_generation.bin",
#     "brio_100_1_seed1": CKPT_DIR +  "/BRIO_Liputan6_checkpoint_100-1-reduceLR/4000/model_generation.bin",
#     "brio_100_1_seed2": CKPT_DIR +  "/BRIO_Liputan6_checkpoint_100-1-second_seed/master_1_100/model_generation.bin",
#     "brio_100_1_seed3": CKPT_DIR +  "/BRIO_Liputan6_checkpoint_100-1-third_seed/master_1_100/model_generation.bin",
# }
SAMPLE_DIR = "/content/drive/MyDrive/BRIO/eval_outputs"
FULL_DIR = "/content/drive/MyDrive/BRIO/eval_outputs_full"


names = ["baseline"] + list(BRIO_CKPTS)
for name in names:
    lines = open(os.path.join(FULL_DIR, name + ".out"), encoding="utf-8").read().splitlines()
    print("%-18s %.1f words per summary" % (name, sum(len(l.split()) for l in lines) / len(lines)))

In [ ]:
ref = open(os.path.join(FULL_DIR, "sample.target"), encoding="utf-8").read().splitlines()
print("reference %.1f words per summary" % (sum(len(l.split()) for l in ref) / len(ref)))

In [ ]:
import pandas as pd
ids = [int(x) for x in open(os.path.join(FULL_DIR, "sample_ids.txt")).read().split()]
lengths = {n: pd.Series([len(l.split()) for l in open(os.path.join(FULL_DIR, n + ".out"), encoding="utf-8").read().splitlines()], index=ids)
           for n in names}
# per-article IndoBERT F1 (use bertscore_mbert.csv for multilingual BERT)
f1 = pd.read_csv(os.path.join(FULL_DIR, "bertscore", "bertscore_indobert-base-p1.csv")).pivot(index="id", columns="system", values="bertscore_F1")

for n in names[1:]:
    d = pd.DataFrame({"len_diff": lengths[n] - lengths["baseline"], "f1_diff": f1[n] - f1["baseline"]})
    same = d[d["len_diff"].abs() <= 2]
    print("%-18s all: %+.4f | similar length (%d articles): %+.4f | corr(length diff, F1 diff): %.2f"
          % (n, d["f1_diff"].mean(), len(same), same["f1_diff"].mean(), d["f1_diff"].corr(d["len_diff"])))